# Latent interpolation

Walks a straight line between the mean latent codes of consecutive taxa,
decoding and rendering a mesh at each step, then stitches the PNGs into one figure.

Edit **Config**, run **Setup** and **Helpers** once, then call `run_render()` and `run_stitch()`.
Endpoints given as an absolute path are encoded with the PointNet encoder instead of
averaged from the training latents, so fossils can be used as endpoints.


## Config

In [ ]:

from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_DIR    = Path.cwd().parent / "run_v72"
CKPT         = "2500"
MODEL_PATH   = f"model/{CKPT}.pth"
LC_PATH      = f"latent_codes/{CKPT}.pth"
ENCODER_CKPT = "encoder/checkpoints/encoder.pt"      # fossil endpoints only

# ── Taxa ──────────────────────────────────────────────────────────────────────
# (label, match): match is a substring of the training filenames, or an absolute
# path to a mesh outside the training set (encoded on the fly, then cached).

# Fig. 1 — lizard to burrower
# Fig. 2 — lizard to snake via the fossil
fig_idx = 1

if fig_idx == 1: 
        OUT_DIR = Path("latent_interpolation/burrower")
        TAXA = [("Cordylosaurus", "gerrhosauridae_cordylosaurus_subtessellatus"),
                ("Chamaesaura",   "cordylidae_chamaesaura_aenea"),
                ("Diplometopon",  "amphisbaenidae_diplometopon_zarudnyi_uf68567")]
        LIFE_HISTORY = {"Cordylosaurus": "SAXICOLOUS",
                         "Chamaesaura":   "GRASS-SWIMMER",
                         "Diplometopon":  "BURROWING"}

else:
        OUT_DIR = Path("latent_interpolation/snake")
        TAXA = [("Varanus",      "varanidae_varanus_komodoensis_tnhc113000"),
                ("Parviraptora", "/home/k.wolcott/NSM/nsm/run_v72/fossils/models_smooth/aligned/ZZZZZ_Fossil_ParviraptorAA_align.vtk"),
                ("Homalopsis",   "homalopsidae_homalopsis_buccata_uf61845")]
        LIFE_HISTORY = {"Varanus":      "LIZARD",
                        "Parviraptora": "FOSSIL",
                        "Homalopsis":   "SNAKE"}

# ── Which vertebrae to average ────────────────────────────────────────────────
REGION     = "T"        # "C" / "T" / "L", or None to pool the whole column
VERT_RANGE = None       # (first, last) inclusive, or None
N_STEPS = 5             # cells per row, endpoints included

# ── Rendering ─────────────────────────────────────────────────────────────────
N_PTS_PER_AXIS = 256    # 128 to draft, 256 for final figures
WIDTH, HEIGHT  = 640, 480
BASE_BG_COL    = [0.98,  0.871, 0.631]   # ramp start, light sand
MAX_BG_COL     = [0.851, 0.361, 0.004]   # ramp end, burnt orange
BASELINE_ROT   = 13     # deg about Z, matches the PC snapshot script

# (label, quadrant of the render_cameras 2x2 panel, extra Z rotation in deg)
VIEWS = [("SIDE", "top_right",   0),
         ("BACK", "bottom_left", 180)]

# ── Fossil encoding ───────────────────────────────────────────────────────────
REFINE_ITERS  = 500     # 0 to use the raw PointNet code
REFINE_LR     = 1e-3
REFINE_LAMBDA = 1e-5

## Setup

In [ ]:

import os, re, gc, sys, copy
import numpy as np
import torch, cv2
import open3d as o3d
import pyvista as pv
from IPython.display import Image

sys.path.insert(0, str(Path.cwd().parent))
from NSM.mesh import create_mesh
from NSM.helper_funcs import (NumpyTransform, pv_to_o3d, load_config,
                              load_model_and_latents, render_cameras,
                              convert_ply_to_vtk)
from NSM.optimization import (build_sdf_dataset, encode_latent_pointnet,
                              optimize_latent_partial, get_top_k_pcs)

os.chdir(TRAIN_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

VOXEL_ORIGIN = (-1.0,) * 3
VOXEL_SIZE   = 2.0 / (N_PTS_PER_AXIS - 1)
ROT_MATRIX   = o3d.geometry.get_rotation_matrix_from_axis_angle(
    [0, 0, np.deg2rad(BASELINE_ROT)])

# (label_a, match_a, label_b, match_b) per consecutive pair
PAIRS = [(TAXA[i][0], TAXA[i][1], TAXA[i + 1][0], TAXA[i + 1][1])
         for i in range(len(TAXA) - 1)]

# One unbroken colour ramp across the whole trajectory: consecutive pairs share
# their endpoint colour, so 2 pairs x 5 steps gives 9 distinct colours, not 10.
BG_COLS = np.linspace(BASE_BG_COL, MAX_BG_COL,
                      N_STEPS * len(PAIRS) - (len(PAIRS) - 1))

config = load_config(config_path="model_params_config.json")
device = config.get("device", "cuda:0")
all_vtk_files = [os.path.basename(f) for f in config["list_mesh_paths"]]

model, _, latent_codes = load_model_and_latents(MODEL_PATH, LC_PATH, config, device)
latents_np = (latent_codes.numpy() if hasattr(latent_codes, "numpy")
              else np.asarray(latent_codes))
_, top_k_reg   = get_top_k_pcs(latent_codes, threshold=0.99)
latent_std_val = torch.tensor(latents_np).std().mean()

# Made once and never torn down -- killing EGL contexts mid-session kills the kernel.
RENDERERS = [o3d.visualization.rendering.OffscreenRenderer(WIDTH, HEIGHT)
             for _ in range(4)]
MATERIAL = o3d.visualization.rendering.MaterialRecord()
MATERIAL.shader     = "defaultLit"
MATERIAL.base_color = [1.0, 1.0, 1.0, 1.0]

print(f"{len(latents_np)} latent codes, dim {latents_np.shape[1]}")
print(f"{len(PAIRS)} row(s): " + "; ".join(f"{a} -> {c}" for a, _, c, _ in PAIRS))

## Helpers

In [ ]:

VERT_PAT = re.compile(r"[_-]([CTL])(\d+)(?!\d)", re.IGNORECASE)

def mean_latent(match_str, region=REGION, vert_range=VERT_RANGE, verbose=True):
    """Mean of every latent whose filename contains match_str and passes the
    region / vertebra-number filters."""
    hits = []
    for i, fname in enumerate(all_vtk_files):
        if match_str.lower() not in fname.lower():
            continue
        m = VERT_PAT.search(fname)
        if region is not None:
            if not m or m.group(1).upper() != region.upper():
                continue
        if vert_range is not None:
            if not m or not (vert_range[0] <= int(m.group(2)) <= vert_range[1]):
                continue
        hits.append(i)

    if not hits:
        raise ValueError(
            f"No meshes matched '{match_str}'"
            + (f" in region {region}" if region else "")
            + (f" vertebrae {vert_range}" if vert_range else "")
            + ". Loosen REGION / VERT_RANGE or check the match string.")
    if verbose:
        print(f"  '{match_str}': {len(hits)} meshes (e.g. {all_vtk_files[hits[0]]})")
    return latents_np[hits].mean(axis=0)

def encode_mesh(mesh_path, verbose=True):
    """Latent for a mesh outside the training set, via PointNet then optional
    refinement. Cached next to the mesh as .latent.npy."""
    npy_path = Path(mesh_path).with_suffix(".latent.npy")
    if npy_path.exists():
        if verbose:
            print(f"  cached latent: {npy_path.name}")
        return np.load(npy_path)

    fname = str(mesh_path)
    if fname.endswith(".ply"):
        _, fname = convert_ply_to_vtk(fname, save=True)

    points, sdf_vals, _, _ = build_sdf_dataset(fname, config, n_samples=None)
    latent = encode_latent_pointnet(os.path.abspath(ENCODER_CKPT),
                                    points, sdf_vals, device)
    if REFINE_ITERS:
        if verbose:
            print(f"  refining {Path(fname).name} for {REFINE_ITERS} iters")
        latent, _ = optimize_latent_partial(
            model, points.squeeze(), sdf_vals, config["latent_size"],
            latent_init=latent, top_k=top_k_reg, iters=REFINE_ITERS,
            lr=REFINE_LR, lambda_reg=REFINE_LAMBDA, clamp_val=None,
            latent_std=latent_std_val, scheduler_step=800, scheduler_gamma=0.7,
            batch_inference_size=32768, multi_stage=True, device=device)

    vec = latent.detach().cpu().numpy().squeeze()
    np.save(npy_path, vec)
    return vec

def crop(panel, quadrant):
    """One camera out of the 2x2 panel render_cameras returns."""
    y = 0 if quadrant.startswith("top") else HEIGHT
    x = 0 if quadrant.endswith("left") else WIDTH
    return panel[y:y + HEIGHT, x:x + WIDTH]

def decode(vec):
    """Latent vector -> Open3D mesh at the baseline orientation."""
    z = torch.tensor(vec, dtype=torch.float32).unsqueeze(0).to(device)
    out = create_mesh(decoder=model, latent_vector=z,
                      n_pts_per_axis=N_PTS_PER_AXIS,
                      voxel_origin=VOXEL_ORIGIN, voxel_size=VOXEL_SIZE,
                      path_original_mesh=None, offset=np.zeros(3), scale=1.0,
                      icp_transform=NumpyTransform(np.eye(4)), objects=1,
                      verbose=False, device=device)
    out = out[0] if isinstance(out, list) else out
    mesh_pv = out if isinstance(out, pv.PolyData) else out.extract_geometry()
    mesh_pv = mesh_pv.compute_normals(cell_normals=False, point_normals=True,
                                      inplace=False, auto_orient_normals=True)
    mesh = pv_to_o3d(mesh_pv)
    mesh.compute_vertex_normals()
    mesh.rotate(ROT_MATRIX, center=mesh.get_center())
    del z, out, mesh_pv
    return mesh

def render_step(vec, bg_col):
    """Decode once, render every view. Returns {label: image}; a view missing
    from the dict failed and is left for the caller to skip."""
    for r in RENDERERS:
        r.scene.set_background(list(bg_col) + [1.0])
    imgs = {}
    try:
        base = decode(vec)
        for lbl, quadrant, deg in VIEWS:
            mesh = copy.deepcopy(base)
            if deg:
                mesh.rotate(o3d.geometry.get_rotation_matrix_from_axis_angle(
                    [0, 0, np.deg2rad(deg)]), center=mesh.get_center())
            panel = render_cameras(RENDERERS, mesh, 0, MATERIAL, 1, n_rotations=1)
            imgs[lbl] = crop(panel, quadrant).copy()
            del mesh, panel
        del base
    except Exception as exc:
        print(f"  render failed: {exc}")
    finally:
        gc.collect()
        if str(device).startswith("cuda"):
            torch.cuda.empty_cache()
    return imgs

def run_render():
    alphas = np.linspace(0.0, 1.0, N_STEPS)

    for row, (lbl_a, match_a, lbl_b, match_b) in enumerate(PAIRS):
        print(f"\n[{row + 1}/{len(PAIRS)}]  {lbl_a} -> {lbl_b}")
        z_a, z_b = [encode_mesh(m) if Path(m).is_absolute() else mean_latent(m)
                    for m in (match_a, match_b)]
        row_dir = OUT_DIR / f"row{row + 1}_{lbl_a}_to_{lbl_b}".replace(" ", "_")
        for lbl, _, _ in VIEWS:
            (row_dir / lbl.lower()).mkdir(parents=True, exist_ok=True)

        row_bg = BG_COLS[row * (N_STEPS - 1):][:N_STEPS]
        for k, alpha in enumerate(alphas):
            imgs = render_step((1 - alpha) * z_a + alpha * z_b, row_bg[k])
            for lbl, _, _ in VIEWS:
                if lbl in imgs:
                    cv2.imwrite(str(row_dir / lbl.lower() /
                                    f"step{k + 1:02d}_of_{N_STEPS}.png"), imgs[lbl])
            print(f"  step {k + 1}/{N_STEPS}  alpha={alpha:.2f}", flush=True)

    print(f"\nPNGs under {OUT_DIR.resolve()}")

def centre_text(img, text, y, scale):
    (tw, _), _ = cv2.getTextSize(text, FONT, scale, THICK)
    cv2.putText(img, text, ((img.shape[1] - tw) // 2, y), FONT, scale,
                TEXT_COL, THICK, cv2.LINE_AA)

def run_stitch(out_name="latent_interpolation_grid.png"):
    blocks = []
    for row, (lbl_a, _, lbl_b, _) in enumerate(PAIRS):
        row_dir = OUT_DIR / f"row{row + 1}_{lbl_a}_to_{lbl_b}".replace(" ", "_")

        cols = []
        for lbl, _, _ in VIEWS:
            files = sorted((row_dir / lbl.lower()).glob("step*.png"))
            if len(files) != N_STEPS:
                print(f"  {row_dir.name}/{lbl.lower()}: {len(files)} PNGs, "
                      f"expected {N_STEPS}")
            if files:
                cols.append(np.vstack([cv2.imread(str(f)) for f in files]))
        if not cols:
            continue

        block = np.hstack(cols)
        centre_text(block, lbl_a.upper(), PAD + 34, LBL_SCALE)
        centre_text(block, LIFE_HISTORY.get(lbl_a, ""), PAD + 34 + LINE_GAP,
                    TRAIT_SCALE)
        y_b = block.shape[0] - PAD
        centre_text(block, lbl_b.upper(), y_b, LBL_SCALE)
        centre_text(block, LIFE_HISTORY.get(lbl_b, ""), y_b - LINE_GAP,
                    TRAIT_SCALE)
        blocks.append(block)

    if not blocks:
        raise FileNotFoundError(f"No PNGs under {OUT_DIR.resolve()} -- "
                                f"run run_render() first.")

    panel = np.hstack(blocks)
    out = OUT_DIR / out_name
    cv2.imwrite(str(out), panel)
    print(f"Saved {out}  ({panel.shape[1]}x{panel.shape[0]} px, "
          f"{len(blocks)} pair(s) x {len(VIEWS)} views x {N_STEPS} steps)")
    return out

## Render
One PNG per step and view, under `OUT_DIR/row<n>_<A>_to_<B>/<view>/`.

In [ ]:
run_render()

## Stitch
Steps run down each column, views side by side, one block per pair.

In [ ]:
FONT        = cv2.FONT_HERSHEY_DUPLEX
LBL_SCALE   = 1.4
TRAIT_SCALE = 1.3
THICK       = 2
LINE_GAP    = 46      # baseline to baseline, taxon name to life history
PAD         = 20
TEXT_COL    = (40, 40, 40)

Image(filename=str(run_stitch()), width=900)